Steps
1. Chunk
2. Embed each chunk with 

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True) 

True

# Prep data - convert PDF to markdown

# Setup Vector database

In [3]:
# DANGER!!! erase DB
import chromadb
client = chromadb.PersistentClient(path="./chroma_db", settings=chromadb.Settings(allow_reset=True))
client.reset() 

True

In [5]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")
for c in client.list_collections():
    col = client.get_collection(c.name)
    print(f"{c.name}: {col.count()} items")

    if c.name =='legal-docs':
        continue
    else:
        peek = col.peek(3)  # first 3 records
        for i, id_ in enumerate(peek["ids"]):
            print(id_, peek["metadatas"][i], peek["documents"][i][:100])

# Query a collection
#col = client.get_collection("your_collection")
#res = col.query(query_texts=["test query"], n_results=3)
#print(res["documents"], res["distances"])

knowledge-base: 0 items


# App

In [1]:
import util
import util.agent

agent = util.agent.build_agent()
agent_response = agent.run("do have any author C clarke stories in memory?")
#print("Agent response:\n", agent_response)

c:\Users\Josiah\Git_Repos\AIAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USER: do have any author C clarke stories in memory?
--- Calling model
AGENT text: Let me check the knowledge base for any stories by Arthur C. Clarke!
AGENT tool call: list_knowledge_base_collections({})
AGENT tool call: search_knowledge_base({'query': 'Arthur C. Clarke story'})
--- Executing tool: list_knowledge_base_collections with input: {}
--- Executing tool: search_knowledge_base with input: {'query': 'Arthur C. Clarke story'}
--- Calling model
AGENT text: Unfortunately, it appears that the knowledge base is currently **empty** — there are no collections or documents stored in it at the moment, including any stories by **Arthur C. Clarke**.

Here's what you could do:

1. **Upload documents** – If you have any Arthur C. Clarke stories (e.g., in `.txt`, `.pdf`, or other formats), you can add them to the knowledge base for me to reference.
2. **Web search** – I can search the web for information *about* his stories, summaries, or publicly available works. Just let me know!

Would y

In [ ]:
for turn in agent.context.turns:
    print(turn)
    for i,step in enumerate(turn.steps):
        print(f"step {i}: {step}")

## Inspect flat-RAG ingestion and search

This demo uses the sample coffee guide and a temporary Chroma database, so it does not alter `./chroma_db`. It defaults to the local embedding backend; set `INSPECTION_EMBEDDING_BACKEND` to `ollama` or `openai` to inspect another configured provider. The local model may download its weights on first use.

In [4]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

from util.document_processor import DocumentProcessor
from util.embedding import EmbeddingProvider
from util.vector_store import VectorDBManager

source_path = Path("data/test_data/coffee_brewing_guide.txt")
collection_name = f"inspection-{uuid4().hex[:8]}"
embedding_backend = os.getenv("INSPECTION_EMBEDDING_BACKEND", "local")


class TracedEmbeddingProvider:
    """Log the vectors produced while delegating to the configured provider."""

    def __init__(self, provider):
        self.provider = provider
        self.last_query_vector = None

    def embed_documents(self, texts):
        print(f"Embedding {len(texts)} chunks")
        vectors = self.provider.embed_documents(texts)
        print(f"Vector dimension: {len(vectors[0])}")
        print(f"First vector (first 8 values): {[round(v, 4) for v in vectors[0][:8]]}")
        return vectors

    def embed_query(self, text):
        print(f"Embedding search query: {text!r}")
        vector = self.provider.embed_query(text)
        self.last_query_vector = vector
        print(f"Query vector dimension: {len(vector)}")
        print(f"Query vector (first 8 values): {[round(v, 4) for v in vector[:8]]}")
        return vector


if not source_path.exists():
    raise FileNotFoundError(source_path.resolve())

print(f"Embedding backend: {embedding_backend}")
base_provider = EmbeddingProvider(backend=embedding_backend)
traced_embeddings = TracedEmbeddingProvider(base_provider)

with TemporaryDirectory(prefix="rag-inspection-") as database_path:
    vector_store = VectorDBManager(
        embedding_provider=traced_embeddings,
        chromadb_path=database_path,
    )
    try:
        processor = DocumentProcessor(
            vector_db_manager=vector_store,
            embedding_provider=traced_embeddings,
        )

        text = processor.read_file(source_path)
        chunks = processor.chunker.split(text, source_path.name)
        print(f"Source: {source_path} ({len(text):,} characters)")
        print(f"Chunks: {len(chunks)}; chunk settings: {processor.chunker.metadata}")
        print(f"First chunk metadata: {chunks[0].metadata}")
        print(f"First chunk preview: {chunks[0].page_content[:300]}...\n")

        success, message = processor.process_document(source_path, collection_name)
        print(f"Ingestion: {message}")
        if not success:
            raise RuntimeError(message)

        collection = vector_store.client.get_collection(collection_name)
        stored = collection.get(include=["documents", "metadatas", "embeddings"])
        stored_vectors = stored["embeddings"]
        print(f"Stored chunks: {len(stored['ids'])}")
        print(f"Stored vector dimension: {len(stored_vectors[0])}")
        print(f"Stored metadata: {stored['metadatas'][0]}")

        query = "What water temperature and coffee-to-water ratio are recommended?"
        matches = vector_store.search_chunks(
            query, k=3, collection_names=[collection_name]
        )
        raw_matches = collection.query(
            query_embeddings=[traced_embeddings.last_query_vector],
            n_results=3,
            include=["documents", "metadatas", "distances"],
        )

        print("\nTop search matches (lower distance is closer):")
        for rank, (document, metadata, distance) in enumerate(
            zip(
                raw_matches["documents"][0],
                raw_matches["metadatas"][0],
                raw_matches["distances"][0],
            ),
            start=1,
        ):
            print(f"\n{rank}. distance={distance:.4f}, source={metadata.get('source')}")
            print(document[:500])
    finally:
        vector_store.client._system.stop()

Embedding backend: local


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4351.73it/s]


Source: data\test_data\coffee_brewing_guide.txt (4,746 characters)
Chunks: 11; chunk settings: {'chunk_size': 512, 'chunk_overlap': 64}
First chunk metadata: {'source': 'coffee_brewing_guide.txt'}
First chunk preview: # Beginner's Guide to Home Brewing Coffee ## Introduction Brewing great coffee at home doesn't require expensive equipment or barista training. This guide covers the four most popular home brewing methods and how to get the best results from each. **The Golden Ratio:** The Specialty Coffee Associati...

Embedding 11 chunks
Vector dimension: 384
First vector (first 8 values): [0.0554, 0.0115, -0.0454, 0.0268, -0.0427, -0.0077, -0.0022, 0.053]
Ingestion: File coffee_brewing_guide.txt processed into 'inspection-ba873b2f'.
Stored chunks: 11
Stored vector dimension: 384
Stored metadata: {'source': 'coffee_brewing_guide.txt'}
Embedding search query: 'What water temperature and coffee-to-water ratio are recommended?'
Query vector dimension: 384
Query vector (first 8 values): [0.